# 비트코인(BTC-USD) 시계열 분석

**기간**: 2023-01-01 ~ 2026-09-29 (일 단위, 수집 목표는 2026-09-30이었으나 수집 시점에 해당 일자의 일봉이 아직 확정되지 않아 2026-09-29까지만 포함됨)
**출처**: Yahoo Finance (`yfinance`) — 개인·비상업 용도 제한

## 실행 순서

1. `pip install -r requirements.txt`
2. `python fetch_data.py` — 데이터를 `data/`에 받는다 (이미 CSV가 있으면 생략 가능)
3. 이 노트북을 위에서 아래로 순서대로 실행

계산 로직은 모두 `btc_analysis.py`에 있고 `tests/`에서 검증된다.
이 노트북은 호출과 출력만 담당한다 — 노트북 셀 안의 계산식은 테스트할 수 없기 때문이다.

In [1]:
import matplotlib
import pandas as pd

import btc_analysis as ba
import plots

plots.setup_korean_font()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

CSV_PATH = "data/btc_usd_2023_2026.csv"
print("준비 완료")

준비 완료


## 1. 데이터 로드 및 기본 정보

기간, 컬럼, 자료형, 기술통계를 먼저 확인한다.

In [2]:
raw = ba.load_raw(CSV_PATH)

print(f"기간: {raw.index.min().date()} ~ {raw.index.max().date()}")
print(f"행 수: {len(raw)}")
print(f"컬럼: {list(raw.columns)}")
print()
raw.info()
print()
display(raw.head())
display(raw.describe().round(2))

기간: 2023-01-01 ~ 2026-09-29
행 수: 1368
컬럼: ['Open', 'High', 'Low', 'Close', 'Volume']

<class 'pandas.DataFrame'>
DatetimeIndex: 1368 entries, 2023-01-01 to 2026-09-29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    1368 non-null   float64
 1   High    1368 non-null   float64
 2   Low     1368 non-null   float64
 3   Close   1368 non-null   float64
 4   Volume  1368 non-null   float64
dtypes: float64(5)
memory usage: 64.1 KB



,Open,High,Low,Close,Volume
Date,,,,,
2023-01-01,16547.914062,16630.439453,16521.234375,16625.080078,9.244362e+09
2023-01-02,16625.509766,16759.343750,16572.228516,16688.470703,1.209778e+10
2023-01-03,16688.847656,16760.447266,16622.371094,16679.857422,1.390308e+10
2023-01-04,16680.205078,16964.585938,16667.763672,16863.238281,1.842174e+10
2023-01-05,16863.472656,16884.021484,16790.283203,16836.736328,1.369276e+10


,Open,High,Low,Close,Volume
count,1368.00,1368.00,1368.00,1368.00,1.368000e+03
mean,66942.86,68085.11,65780.22,66992.13,3.597892e+10
std,28947.36,29352.12,28464.05,28918.78,2.261166e+10
min,16547.91,16630.44,16521.23,16625.08,5.331173e+09
25%,40947.94,41872.26,39940.39,41257.50,1.902292e+10
50%,66913.47,68188.89,65581.63,66928.29,3.060859e+10
75%,90512.02,91875.95,88726.51,90514.42,4.620441e+10
max,124752.14,126198.07,123196.05,124752.53,1.817464e+11


## 2. 결측치 / 이상치 점검

**처리 기준**

- **데이터 오류는 제거한다**: 가격 0 이하, `High < Low`, 전일 대비 ±50% 초과
- **결측 날짜는 ffill로 채운다**: 거래 기록이 없는 날은 직전 가격이 유지된 것으로 본다
- **보간된 날이 관여한 수익률은 제외한다**: 가짜 0% 수익률이 섞이면 변동성이 과소추정된다
- **극단 수익률은 제거하지 않는다**: 비트코인의 하루 ±10% 변동은 측정 오류가 아니라 분석 대상 신호다

아래 결과는 기준을 적용한 뒤의 실제 수치다 — 수집된 데이터가 이미 완전하고 연속적이어서
0건이 나오는 것이 정상이며, 이는 "점검했지만 고칠 것이 없었다"는 결과다.

In [3]:
invalid = ba.find_invalid_rows(raw)
print(f"데이터 오류 행: {len(invalid)}건")
if len(invalid):
    display(invalid)

clean = ba.drop_invalid_rows(raw)
reindexed, missing = ba.reindex_daily(clean)
print(f"전체 날짜 재색인 후 행 수: {len(reindexed)}")
print(f"원본에 없던 날짜: {len(missing)}건")
if len(missing):
    print(list(missing[:20]))

df = ba.fill_prices(reindexed)
print(f"ffill로 보간한 날짜: {int(df['is_filled'].sum())}건")
print(f"보간 후 종가 결측: {int(df['Close'].isna().sum())}건")

데이터 오류 행: 0건
전체 날짜 재색인 후 행 수: 1368
원본에 없던 날짜: 0건
ffill로 보간한 날짜: 0건
보간 후 종가 결측: 0건


## 3. 시계열 지표 계산

- **이동평균 7/30/90일** — 단기 노이즈를 걷어내 추세를 분리한다
- **일별·누적 변화율** — 가격 레벨이 다른 구간을 공정하게 비교한다
- **30일 롤링 변동성(연율화)** — 변동성의 시간 변화를 측정한다. 연중무휴 거래이므로 `sqrt(365)`로 연율화한다
- **최대 낙폭(MDD)** — 하락 리스크 규모를 단일 수치로 요약한다

In [4]:
df = ba.add_returns(df)
df = ba.add_moving_averages(df, windows=(7, 30, 90))
df = ba.add_volatility(df, window=30)

print(f"시작가 ${df['Close'].iloc[0]:,.0f}  종료가 ${df['Close'].iloc[-1]:,.0f}")
print(f"전체 누적 수익률: {df['cum_return'].iloc[-1]*100:,.1f}%")
print(f"최고가 ${df['Close'].max():,.0f} ({df['Close'].idxmax().date()})")
print(f"최저가 ${df['Close'].min():,.0f} ({df['Close'].idxmin().date()})")
print()
print(f"일평균 수익률: {df['daily_return'].mean()*100:.4f}%")
print(f"일간 표준편차: {df['daily_return'].std()*100:.2f}%")
print(f"연율 변동성 — 평균 {df['vol_30'].mean()*100:.1f}%  최대 {df['vol_30'].max()*100:.1f}% ({df['vol_30'].idxmax().date()})  최소 {df['vol_30'].min()*100:.1f}% ({df['vol_30'].idxmin().date()})")
print()

mdd = ba.max_drawdown(df["Close"])
print(f"최대 낙폭: {mdd['mdd']*100:.1f}%  ({mdd['peak_date'].date()} 고점 → {mdd['trough_date'].date()} 저점)")
print()

outliers = ba.detect_return_outliers(df, z_threshold=3.0)
print(f"3시그마 초과 일자: {len(outliers)}일 (전체의 {len(outliers)/df['daily_return'].notna().sum()*100:.1f}%)")
print("--- 하락 상위 5일 ---")
display(outliers.nsmallest(5, "daily_return"))
print("--- 상승 상위 5일 ---")
display(outliers.nlargest(5, "daily_return"))

시작가 $16,625  종료가 $83,622
전체 누적 수익률: 403.0%
최고가 $124,753 (2025-10-06)
최저가 $16,625 (2023-01-01)

일평균 수익률: 0.1478%
일간 표준편차: 2.44%
연율 변동성 — 평균 44.7%  최대 84.6% (2024-03-26)  최소 17.0% (2023-08-13)

최대 낙폭: -53.1%  (2025-10-06 고점 → 2026-06-30 저점)

3시그마 초과 일자: 20일 (전체의 1.5%)
--- 하락 상위 5일 ---


,Close,daily_return,z_score
Date,,,
2026-02-05,62702.097656,-0.141299,-5.845592
2025-03-03,86065.671875,-0.086820,-3.615131
2024-03-19,61912.773438,-0.083434,-3.476468
2024-01-12,42853.167969,-0.075815,-3.164534
2023-01-20,22676.552734,0.075391,3.026141


--- 상승 상위 5일 ---


,Close,daily_return,z_score
Date,,,
2026-02-06,70555.390625,0.125248,5.067364
2024-08-08,61710.136719,0.121443,4.911574
2023-10-23,33086.234375,0.103099,4.160548
2024-11-11,88701.484375,0.102235,4.125187
2024-03-20,67913.671875,0.096925,3.907777


## 4. 패턴 탐색: 월별 / 요일별

집계 단위를 바꾸는 이유: 일 단위에서는 노이즈가 커서 반복 패턴이 보이지 않는다.
월·요일로 묶어 같은 구간이 연도를 건너도 일관된지 확인한다.

In [5]:
pivot = ba.monthly_return_pivot(df)
print("=== 월별 수익률 (%) ===")
display((pivot * 100).round(1))

print("=== 월별 평균 (%) ===")
display((pivot.mean() * 100).round(1).to_frame("평균 수익률(%)").T)

print("=== 월별 부호 일관성 (양수였던 연도 수 / 데이터가 있는 연도 수) ===")
consistency = pivot.apply(lambda c: f"{int((c > 0).sum())}/{int(c.notna().sum())}")
display(consistency.to_frame("양수 연도").T)

print("=== 요일별 수익률 ===")
weekday = ba.weekday_return_table(df)
display((weekday[["mean", "median", "std"]] * 100).round(3).join(weekday[["count"]]))

=== 월별 수익률 (%) ===


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2023,39.2,0.0,23.0,2.8,-7.0,12.0,-4.1,-11.3,4.0,28.6,8.8,12.1
2024,0.8,43.7,16.6,-15.0,11.3,-7.1,3.1,-8.7,7.4,10.9,37.4,-3.1
2025,9.6,-17.6,-2.2,14.1,11.1,2.4,8.0,-6.5,5.4,-3.9,-17.5,-3.2
2026,-10.2,-14.8,1.8,11.8,-3.6,-20.4,7.3,25.1,6.5,NaN,NaN,NaN


=== 월별 평균 (%) ===


month,1,2,3,4,5,6,7,8,9,10,11,12
평균 수익률(%),9.8,2.8,9.8,3.4,3.0,-3.3,3.6,-0.4,5.8,11.8,9.6,1.9


=== 월별 부호 일관성 (양수였던 연도 수 / 데이터가 있는 연도 수) ===


month,1,2,3,4,5,6,7,8,9,10,11,12
양수 연도,3/4,2/4,3/4,3/4,2/4,2/4,3/4,1/4,4/4,2/3,2/3,1/3


=== 요일별 수익률 ===


,mean,median,std,count
월,0.437,0.229,2.897,196
화,0.000,-0.160,2.464,196
수,0.419,0.035,2.765,195
목,-0.279,-0.357,2.601,195
금,0.227,0.105,2.712,195
토,0.034,0.036,1.202,195
일,0.196,0.061,1.968,195


## 5. 베이스라인 예측 (보너스)

마지막 30일을 홀드아웃으로 두고, 그 이전 데이터만으로 3개 베이스라인을 비교한다.

**방향 정확도 주의**: 방향 정확도는 예측 시작 시점(`origin_value`, 학습 구간 마지막 종가)
대비 상승/하락 부호가 실제와 일치한 비율이다.

1. **Naive만 구조적으로 0%가 나온다.** Naive의 예측값은 `origin_value`와 정확히 같으므로
   부호가 항상 0이고, 실제 부호는 +1 또는 -1이므로 둘이 일치할 수 없다.
2. **이동평균과 Drift는 다르다.** 두 모델은 `origin_value`와 다른 수준(각각 학습 구간
   마지막 7일 평균, 추세 연장값)을 예측하므로, 부호가 +1 또는 -1 중 하나로 예측 구간
   전체에서 고정된 상수가 된다. 실제로 이번 실행에서는 Naive 0.0%, 이동평균 70.0%로
   나타나 이동평균이 0%가 아님을 보여준다.
3. **그 결과 이 지표는 일별 방향 예측 능력이 아니라 "예측 기간 전체에 대한 단 한 번의
   방향 베팅"이 맞았는지를 측정한다.** 따라서 이 지표만으로 모델 간 우열을 판단해서는
   안 되며, 모델 순위는 MAE·MAPE로 판단한다.

In [6]:
predictions, scores = ba.run_baselines(df["Close"], horizon=30)

print(f"홀드아웃: {predictions.index.min().date()} ~ {predictions.index.max().date()} ({len(predictions)}일)")
actual_change = predictions["actual"].iloc[-1] / predictions["actual"].iloc[0] - 1
print(f"홀드아웃 구간 실제 변화: {actual_change*100:+.1f}%")
print()
display(scores.round(2))
print(f"MAE 최저 모델: {scores['mae'].idxmin()}")
print()

plots.plot_price_trend(df, "images/01_price_trend_ma.png")
plots.plot_returns_volatility(df, "images/02_returns_volatility.png")
plots.plot_monthly_heatmap(pivot, "images/03_monthly_heatmap.png")
plots.plot_weekday_box(df, "images/04_weekday_boxplot.png")
plots.plot_baseline_forecast(predictions, scores, "images/05_baseline_forecast.png")
print("그래프 5개 저장 완료 → images/")

홀드아웃: 2026-08-31 ~ 2026-09-29 (30일)
홀드아웃 구간 실제 변화: +6.5%



,mae,mape,direction_accuracy
Naive,3191.70,3.85,0.0
이동평균(7일),2877.90,3.50,70.0
Drift,2819.27,3.42,70.0


MAE 최저 모델: Drift



그래프 5개 저장 완료 → images/
